# 01 — Data audit · Crowdfund Compass (Group 11)
Goal: load the merged Kickstarter table, re-verify the headline facts, and check the known data issues **before** any modelling.

Workflow: 1) Setup → 2) Import → 3) Verify facts → 4) Known issues → 5) Sanity checks → 6) Conclusions.

*AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup
Load the libraries. In Colab, mount Google Drive and point `path` to the `merged_v2` folder (built by `merge_v2.py`).

In [1]:
import glob
import pandas as pd
import numpy as np
pd.set_option('display.width', 200)

# from google.colab import drive; drive.mount('/content/drive')
path = 'drive/My Drive/ks_scrapes/merged_v2'   # folder with kickstarter_merged_v2_part1..8.parquet and presence_part1..4.parquet

## 2) Import the data
The merged table (version 2, rebuilt 16.11.2026) is split into 8 files. We stack them into one table. One row = one Kickstarter project in its latest observed state.

In [2]:
files = sorted(glob.glob(path + '/kickstarter_merged_v2_part*.parquet'))
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(len(files), 'files |', len(df), 'rows |', df.id.nunique(), 'unique project ids')

# Readable dates (the raw columns are unix seconds, UTC)
df['launch_date'] = pd.to_datetime(df.launched_at, unit='s')
df['deadline_date'] = pd.to_datetime(df.deadline, unit='s')
df['launch_year'] = df.launch_date.dt.year
df.state.value_counts()

8 files | 488967 rows | 488967 unique project ids


state
successful    262375
failed        167707
canceled       22222
live           21913
submitted      11055
started         2658
suspended       1037
Name: count, dtype: int64

## 3) Verify the headline facts
**Finished** = state is `successful` or `failed`. Success rate = successful / finished.

In [3]:
fin = df[df.state.isin(['successful', 'failed'])].copy()
fin['success'] = (fin.state == 'successful').astype(int)
print('finished projects:', len(fin))
print('success rate overall:', round(fin.success.mean(), 4))
print('success rate 2014-2017:', round(fin[fin.launch_year.between(2014, 2017)].success.mean(), 4))
print('subcategories:', fin.subcategory.nunique(), '| countries:', fin.country.nunique())
print('parent_category values:', fin.parent_category.nunique(), '(15 expected)')
per_creator = fin.groupby('creator_id').size()
print('creators:', len(per_creator), '| with more than one finished project:', (per_creator > 1).sum())
print('prelaunch_activated known:', round(fin.prelaunch_activated.notna().mean(), 4))

finished projects: 430082
success rate overall: 0.6101
success rate 2014-2017: 0.4363
subcategories: 164 | countries: 25
parent_category values: 15 (15 expected)
creators: 323236 | with more than one finished project: 43923
prelaunch_activated known: 0.5933


## 4) Known issues
### 4.1 Success bias
Success rate and number of finished projects by launch year. Benchmark (updated 26.11.2026, the same as in the report and the slides): Kickstarter's official stats page of 1 January 2026, all-time: 292,062 funded, 391,396 unsuccessful, 1,939 live, success rate 42.73%. Kickstarter's "unsuccessful" is everything launched that is neither funded nor live, so it includes cancelled and suspended projects. We therefore show two comparisons: our failed projects only (the 42% used so far) and, like with like, our failed + cancelled + suspended projects.

In [4]:
print(fin.groupby('launch_year').success.agg(n='size', success_rate='mean').round(3))

# How much of Kickstarter do we hold? Projects that ended before 1 January 2026 against the official totals of that day.
OFF_FUNDED, OFF_UNSUCCESSFUL, OFF_LIVE, OFF_LAUNCHED = 292062, 391396, 1939, 685397
print('official: launched = funded + unsuccessful + live?', OFF_LAUNCHED == OFF_FUNDED + OFF_UNSUCCESSFUL + OFF_LIVE)
pre = df[(df.deadline_date < '2026-01-01') & (df.launched_at > 1e9)]
n = pre.state.value_counts()
ours_s, ours_f = n.get('successful', 0), n.get('failed', 0)
ours_cs = n.get('canceled', 0) + n.get('suspended', 0)
print('funded held  :', ours_s, '=', round(ours_s / OFF_FUNDED, 3), 'of the official count')
print('failed held  :', ours_f, '=', round(ours_f / OFF_UNSUCCESSFUL, 3), 'of the official unsuccessful count (failed only)')
print('failed + cancelled + suspended held:', ours_f + ours_cs, '=', round((ours_f + ours_cs) / OFF_UNSUCCESSFUL, 3), '(like with like)')
print('our success rate:', round(ours_s / (ours_s + ours_f), 3), 'vs official', round(OFF_FUNDED / (OFF_FUNDED + OFF_UNSUCCESSFUL), 3))

# Spot check: Kickstarter reported 5,314 of 6,646 tabletop projects funded in 2024 (80%)
t = fin[(fin.subcategory == 'Tabletop Games') & (fin.launch_year == 2024)]
print('Tabletop 2024 in our data:', len(t), 'projects,', t.success.sum(), 'funded, rate', round(t.success.mean(), 3))

                 n  success_rate
launch_year                     
2009           878         0.659
2010          6570         0.699
2011         15124         0.700
2012         24285         0.630
2013         24924         0.648
2014         40224         0.441
2015         47356         0.379
2016         34463         0.460
2017         26538         0.500
2018         25484         0.634
2019         27956         0.692
2020         25027         0.735
2021         22539         0.779
2022         19397         0.736
2023         18434         0.705
2024         25471         0.711
2025         26533         0.708
2026         18879         0.778
official: launched = funded + unsuccessful + live? True


funded held  : 246842 = 0.845 of the official count
failed held  : 162910 = 0.416 of the official unsuccessful count (failed only)
failed + cancelled + suspended held: 185448 = 0.474 (like with like)
our success rate: 0.602 vs official 0.427
Tabletop 2024 in our data: 3742 projects, 3517 funded, rate 0.94


**Interpretation:** we hold most successful projects but well under half of the failed ones, and the gap is larger in recent years. Raw success rates in this data are too high and must never be presented as Kickstarter's rates.

### 4.2 `has_video` and `prelaunch_activated` by scrape
Scrape files before 2023-12-14 do not contain these fields. Version 2 of the merge leaves them **missing** there (version 1 wrote `False` for video, which was wrong).

In [5]:
by_scrape = df.groupby('scrape').agg(rows=('id', 'size'), video_known=('has_video', lambda s: s.notna().mean()),
                                    has_video_share=('has_video', 'mean'),
                                    prelaunch_known=('prelaunch_activated', lambda s: s.notna().mean()))
print(by_scrape.round(3))

# Rule: both fields are only KNOWN for rows whose latest scrape is 2023-12-14 or later
fin['presentation_known'] = fin.has_video.notna() & fin.prelaunch_activated.notna()
print(fin.groupby('launch_year').presentation_known.mean().round(3))

# Is 'known' neutral? No: it depends on the outcome, so it must never be a feature
print(fin[fin.launch_year.between(2010, 2022)].groupby(['launch_year', 'presentation_known']).success.mean().unstack().round(3))

             rows  video_known  has_video_share  prelaunch_known
scrape                                                          
2016-03-22   6266          0.0              NaN              0.0
2016-09-15   9299          0.0              NaN              0.0
2017-03-15  30910          0.0              NaN              0.0
2017-09-15   3321          0.0              NaN              0.0
2018-03-15   4507          0.0              NaN              0.0
2018-09-13  45533          0.0              NaN              0.0
2019-06-13  11568          0.0              NaN              0.0
2019-12-12  11747          0.0              NaN              0.0
2020-06-18  11013          0.0              NaN              0.0
2020-12-17  23433          0.0              NaN              0.0
2021-06-17   3560          0.0              NaN              0.0
2021-12-09   1252          0.0              NaN              0.0
2022-06-09  35729          0.0              NaN              0.0
2022-12-15   1342        

**Interpretation:** both fields are known only for projects last listed in 2023-12 or later, and for all launches from 2023 on (one exception). Whether they are known depends on the outcome (e.g. 2012 launches: 82% funded if known, 57% if unknown). Rule: these two fields, and any "missing" flag for them, are used only in a model trained on launches from 2023 on.

### 4.3 Coverage of 2021–2023

In [6]:
print(df.groupby('scrape').size())   # rows whose LATEST observation is this scrape
print(fin[fin.launch_year.between(2019, 2026)].groupby('launch_year').size())

scrape
2016-03-22     6266
2016-09-15     9299
2017-03-15    30910
2017-09-15     3321
2018-03-15     4507
2018-09-13    45533
2019-06-13    11568
2019-12-12    11747
2020-06-18    11013
2020-12-17    23433
2021-06-17     3560
2021-12-09     1252
2022-06-09    35729
2022-12-15     1342
2023-06-08       25
2023-12-14     1782
2024-06-20     7937
2024-12-12    27868
2025-06-12    14569
2025-12-18    12937
2026-06-11    14762
2026-08-12    86922
2026-10-12    27562
2026-11-16    95123
dtype: int64


launch_year
2019    27956
2020    25027
2021    22539
2022    19397
2023    18434
2024    25471
2025    26533
2026    18879
dtype: int64


**Interpretation:** several scrapes are small, but because later scrapes re-list older projects, each launch year 2021–2023 still has 18–23k finished projects.

### 4.4 Exclusions

In [7]:
print('not finished (dropped from training):', (~df.state.isin(['successful', 'failed'])).sum())
print('impossible launch date (before Kickstarter existed):', (df.launch_date < '2009-04-01').sum(),
      '| of which finished:', (fin.launch_date < '2009-04-01').sum())
last_scrape = pd.Timestamp('2026-11-16')
print('finished but launched in the last 60 days (only short campaigns can have ended):',
      (fin.launch_date > last_scrape - pd.Timedelta(days=60)).sum())

not finished (dropped from training): 58885
impossible launch date (before Kickstarter existed): 13723 | of which finished: 0
finished but launched in the last 60 days (only short campaigns can have ended): 1806


### 4.5 `parent_category` (fixed in version 2)
Older scrapes have no parent name; version 1 copied the subcategory. Version 2 takes the parent from the category slug (`art/painting` → Art).

In [8]:
print(fin.groupby('parent_category').size())
print('category ids:', fin.cat_id.nunique(), '| subcategory names:', fin.subcategory.nunique(), '(7 names exist under several parents, so use cat_id / cat_slug as the key)')

parent_category
Art             40193
Comics          25653
Crafts           9543
Dance            3716
Design          28401
Fashion         24768
Film & Video    55146
Food            24244
Games           53650
Journalism       5636
Music           46225
Photography     10073
Publishing      50282
Technology      43537
Theater          9015
dtype: int64
category ids: 174 | subcategory names: 164 (7 names exist under several parents, so use cat_id / cat_slug as the key)


### 4.6 Where do the failed projects get lost?
The presence table has one row per project and scrape in which it was listed.

In [9]:
pres = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(path + '/presence_part*.parquet'))], ignore_index=True)
pres['S'] = pd.to_datetime(pres.scrape); pres['D'] = pd.to_datetime(pres.deadline, unit='s')

# (a) Inside one scrape: does the share of failed projects fall with the time since they ended?
q = pres[(pres.scrape == '2025-12-18') & pres.state.isin(['successful', 'failed'])].copy()
q['months_since_end'] = (q.S - q.D).dt.days // 30
t = q[q.months_since_end.between(0, 12)].groupby(['months_since_end', 'state']).size().unstack()
t['failed_share'] = (t.failed / (t.failed + t.successful)).round(3)
print(t)

# (b) Once listed as finished: are failed projects dropped faster from the next scrape?
A = pres[(pres.scrape == '2025-06-12') & pres.state.isin(['successful', 'failed'])]
print(A.assign(kept=A.id.isin(set(pres[pres.scrape == '2025-12-18'].id))).groupby('state').kept.mean().round(3))

# (c) Projects seen LIVE: do they come back with a final state in a later scrape?
A = pres[(pres.scrape == '2025-06-12') & (pres.state == 'live')].copy()
later = pres[(pres.scrape > '2025-06-12') & pres.state.isin(['successful', 'failed', 'canceled', 'suspended'])]
A['seen_final'] = A.id.isin(set(later.id))
A['funded_share'] = A.pledged / A.goal; A['days_left'] = (A.D - A.S).dt.days
A['group'] = np.select([A.funded_share >= 1, (A.funded_share < 0.2) & (A.days_left <= 7)],
                       ['already funded', 'under 20% funded, 7 days or less left'], 'other')
print(A.groupby('group').seen_final.agg(n='size', seen_final='mean').round(3))

state             failed  successful  failed_share
months_since_end                                  
0                    692        1818         0.276
1                    668        1953         0.255
2                    650        1635         0.284
3                    634        1479         0.300
4                    681        1491         0.314
5                    690        1648         0.295
6                    643        1606         0.286
7                    580        1454         0.285
8                    631        1407         0.310
9                    639        1483         0.301
10                   531        1090         0.328
11                   549         797         0.408
12                   659        1357         0.327


state
failed        0.924
successful    0.941
Name: kept, dtype: float64


                                          n  seen_final
group                                                  
already funded                         1222       0.908
other                                  1839       0.716
under 20% funded, 7 days or less left   203       0.680


**Interpretation:** failed projects are not dropped over time: their share is flat for 12 months (a) and they are kept from scrape to scrape like successful ones (b). They are lost at the moment the campaign ends: an already-funded live campaign reappears as finished in about 91% of cases, an almost certainly failing one in about 68% (c). Many more failures are never listed at all.

## 5) Sanity checks on the pre-launch fields

In [10]:
fin['goal_usd'] = fin.goal * fin.static_usd_rate
fin['duration_days'] = (fin.deadline - fin.launched_at) / 86400
fin['prep_days'] = (fin.launched_at - fin.created_at) / 86400
print(fin[['goal_usd', 'duration_days', 'prep_days']].describe(percentiles=[.01, .5, .99]).round(1))
print('goal <= 1 USD:', (fin.goal_usd <= 1).sum(), '| goal >= 10m USD:', (fin.goal_usd >= 1e7).sum(),
      '| duration > 92 days:', (fin.duration_days > 92).sum())
print('state contradicts pledged vs goal:', ((fin.success == 1) & (fin.pledged < fin.goal)).sum()
      + ((fin.success == 0) & (fin.pledged >= fin.goal)).sum())
print('missing: blurb', fin.blurb.isna().sum(), '| creator_id', fin.creator_id.isna().sum(), '| city', fin.city.isna().sum())

          goal_usd  duration_days  prep_days
count     430082.0       430082.0   430082.0
mean       31041.4           32.9       55.0
std       933612.6           12.7      149.2
min            0.0            1.0        0.0
1%            57.3            7.0        0.0
50%         4500.0           30.0       14.8
99%       235202.1           60.0      665.1
max    152350076.0          120.6     5061.8
goal <= 1 USD: 437 | goal >= 10m USD: 127 | duration > 92 days: 6
state contradicts pledged vs goal: 11
missing: blurb 6 | creator_id 784 | city 1007


## 6) Conclusions
1. The headline facts are confirmed. Version 2 of the merge fixes `parent_category` (15 values).
2. Success bias is large: against Kickstarter's totals of 1 January 2026 we hold about 85% of all funded projects but only about 42% (failed only) or 47% (failed, cancelled or suspended, like with like) of the unsuccessful ones.
3. `has_video` and `prelaunch_activated` are unknown for every project last seen before 2023-12, and "known" depends on the outcome: use them only in a model trained on launches from 2023 on.
4. Coverage by launch year is adequate; drop the last 60 days of launches from the test set.
5. Extreme goals and durations are flagged, not deleted.
6. Failed projects are lost when the campaign ends, not afterwards.